# Machine Learning — Lab 1
## ML Problem Formulation, Data Splits, and Baselines

**Course Learning Outcomes — CLO1 and CLO5**  
- **CLO1:** Explain the fundamental concepts of machine learning, including learning paradigms, data representation, model assumptions, and evaluation principles.  
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`  
**Submission:** completed notebook containing predictions, code, split reports, justifications, debugging answers, experimental results, and reflection.

> **Assessment principle:** Working code is only part of the evidence. Most marks come from your ability to **formulate the problem, predict before running, justify data-split decisions, detect leakage, establish a baseline, and explain what the results mean**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Objectives & problem formulation | 10 min | Identify learning paradigm, features, target, and prediction goal |
| 2. Inspect a real dataset | 15 min | Understand examples, features, classes, and class balance |
| 3. Build train/validation/test splits | 30 min | Create a valid, personalized 60/20/20 split |
| 4. Construct and evaluate a baseline | 25 min | Implement a majority-class baseline and interpret accuracy |
| 5. Leakage, experiment & analysis | 25 min | Diagnose invalid workflows and compare alternative splits |
| 6. Challenge, reflection & viva | 15 min | Formulate an assigned scenario and defend your reasoning |

> **Main idea:** Before choosing an algorithm, define **what is being learned**, **what information the model may use**, and **how unseen performance will be evaluated**.

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish **supervised**, **unsupervised**, and **reinforcement learning** scenarios;
2. distinguish **classification** from **regression**;
3. identify **observations, features, targets, and classes** in a dataset;
4. explain the roles of **training, validation, and test data**;
5. create reproducible **stratified** train/validation/test splits;
6. construct and evaluate a simple **baseline model**;
7. explain why model performance should be compared with a baseline;
8. identify **target leakage**, **test-set tuning**, and **split contamination**;
9. predict the consequences of changing a split before running code;
10. formulate a new ML problem using a structured problem specification.

## Before You Start

This lab deliberately uses a simple baseline instead of a sophisticated classifier.

The goal is to learn the **experimental discipline** that every later ML model will use:

$$
\text{Problem}
\rightarrow
\text{Data}
\rightarrow
\text{Split}
\rightarrow
\text{Training}
\rightarrow
\text{Validation}
\rightarrow
\text{Final Test}.
$$

### Required working style

Throughout the notebook, you will be asked to:

- **predict** an outcome before executing code;
- explain **why** a split or baseline is valid;
- identify deliberately incorrect workflows;
- modify one parameter and explain the effect;
- answer short individual-understanding questions.

Do not delete your predictions after you run the code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("Machine Learning Lab 1 environment ready.")

# Part I — Formulate the Machine Learning Problem

A machine-learning problem should be defined before a model is selected.

For supervised learning, a useful formulation identifies:

$$
(X,\;y,\;\text{task},\;\text{evaluation}).
$$

where:

- $X$ contains the input **features**;
- $y$ contains the **target/label**;
- the task may be classification or regression;
- evaluation defines what successful generalization means.

For unsupervised learning, there may be no target $y$.

## Task 1.1 — Classify the Learning Scenarios

Complete the table **before running any model code**.

| Scenario | Learning paradigm | Classification / Regression / Other | Why? |
|---|---|---|---|
| Predict monthly apartment rent from area, district, and number of rooms |  |  |  |
| Group customers according to purchasing behavior without predefined customer types |  |  |  |
| Predict whether an email is spam from labeled historical emails |  |  |  |
| A robot improves navigation by receiving rewards for successful actions |  |  |  |
| Predict tomorrow's electricity demand in kWh from historical measurements |  |  |  |

### Your reasoning

For each row, justify the choice using the presence or absence of **labels**, the type of **target**, or the use of **rewards**.

## Task 1.2 — Identify Features and Targets

Consider a system that predicts whether a student will submit an assignment on time.

Available variables:

- number of LMS logins in the previous week;
- number of previous late submissions;
- attendance percentage;
- assignment difficulty category;
- final submission timestamp;
- whether the assignment was submitted on time.

Answer:

1. Which variable should be the **target**?
2. Which variables could be legitimate **features** if the prediction is made 48 hours before the deadline?
3. Which variable is obviously unavailable at prediction time?
4. Is the task classification or regression?
5. What would one training example contain?

**Your answers:**

## Task 1.3 — Parameters vs. Hyperparameters vs. Data Decisions

Classify each item.

| Item | Parameter / Hyperparameter / Data decision / Other |
|---|---|
| Learned coefficient in a regression model |  |
| Number of neighbors $k$ in KNN |  |
| Choosing a 60/20/20 split |  |
| Maximum depth of a decision tree |  |
| One student's feature vector |  |
| Learned neural-network weight |  |

> You are not expected to know the later algorithms in detail yet. Focus on whether the quantity is **learned from training data**, **chosen by the practitioner**, or is part of the **dataset/workflow**.

# Part II — Load and Inspect a Real Dataset

We will use the **Iris flower dataset** included with `scikit-learn`.

Each observation is an iris flower described by four numerical measurements:

- sepal length;
- sepal width;
- petal length;
- petal width.

The original dataset contains three iris species. In this lab, we create a simple binary task:

> **Predict whether a flower is _Iris virginica_ or not.**

This gives us an intuitive **supervised binary-classification** problem and a useful majority-class baseline.

In [ ]:
iris = load_iris(as_frame=True)

X = iris.data.copy()

# Binary target for this lab:
# 1 = Iris virginica
# 0 = not Iris virginica
y = (iris.target == 2).astype(int)
y.name = "is_virginica"

class_names = ["not_virginica", "virginica"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])
print("Feature names:", list(X.columns))
print("Class names:", class_names)

## Task 2.1 — Interpret the Dataset Dimensions

From the output above, answer:

1. How many observations are available?
2. How many features describe each observation?
3. How many possible target classes exist?
4. What does one row of `X` represent?
5. What does the corresponding value in `y` represent?
6. Why is `y` not included inside `X`?

**Your answers:**

In [ ]:
display(X.head())

print("\nFirst five binary target labels:")
print(y.head().to_list())

print("\nCorresponding class names:")
print([class_names[label] for label in y.head().to_list()])

## Task 2.2 — Inspect Feature Meaning

Choose **three** feature columns from the displayed dataset.

For each:

1. state whether it is numerical or categorical;
2. describe what one value represents;
3. state whether the feature would later require categorical encoding;
4. state whether its numerical scale might matter for some algorithms.

| Feature | Type | Meaning | Encoding needed? | Could scaling matter? |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |

## Task 2.3 — Predict Class Balance Before Execution

Before running the next cell:

1. Do you expect the two classes (`virginica` and `not_virginica`) to be perfectly balanced?
2. Which class do you expect to be more frequent?
3. What accuracy would a classifier achieve if it **always predicted the majority class**?
4. Would such a classifier actually be learning useful flower-measurement patterns?

Write your prediction first.

**Your prediction:**

In [ ]:
class_counts = y.value_counts().sort_index()
class_table = pd.DataFrame({
    "class_id": class_counts.index,
    "class_name": [class_names[i] for i in class_counts.index],
    "count": class_counts.values,
})
display(class_table)

plt.figure(figsize=(7, 4))
plt.bar(class_table["class_name"], class_table["count"])
plt.xlabel("Binary Iris class")
plt.ylabel("Number of observations")
plt.title("Class Distribution")
plt.show()

full_majority_rate = class_counts.max() / len(y)
print(f"Full-dataset majority-class proportion: {full_majority_rate:.3f}")

## Task 2.4 — Interpret the Class Distribution

Answer:

1. Which class is most frequent?
2. What is the majority-class proportion?
3. Why is this value useful as a **reference**, but not yet the final baseline used for validation/test evaluation?
4. Why should the baseline class ultimately be determined from the **training labels only**?

**Your answers:**

# Part III — Create Training, Validation, and Test Sets

We will use:

$$
60\% \text{ training},\qquad
20\% \text{ validation},\qquad
20\% \text{ test}.
$$

The roles are different:

- **Training set:** learn model parameters.
- **Validation set:** compare model choices and hyperparameters.
- **Test set:** estimate final performance after all model decisions are complete.

We will also use **stratification** so that class proportions remain approximately similar across the three sets.

## Task 3.1 — Personalize Your Experiment

Enter the **last four digits** of your student ID below.

Your digits determine the random seed used for the split. This gives each student a reproducible but slightly different train/validation/test assignment.

Do not use another student's value.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID before continuing.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 1000 + (STUDENT_ID_LAST4 % 9000)

print("Your reproducible split seed is:", SEED)

## Task 3.2 — Predict the Split Sizes

Before running the splitting code, use the dataset size from Part II.

Predict approximately:

- number of training observations;
- number of validation observations;
- number of test observations.

Also predict whether all three sets should contain examples from all two classes.

| Split | Predicted number of observations | Should contain all classes? |
|---|---:|---|
| Training |  |  |
| Validation |  |  |
| Test |  |  |

**Your reasoning:**

In [ ]:
# First split: 60% training, 40% temporary.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=SEED,
    stratify=y,
)

# Second split: divide the temporary 40% equally -> 20% validation, 20% test.
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)
print("Total observations:", len(X_train) + len(X_valid) + len(X_test))

## Task 3.3 — Compare Prediction with Reality

1. Record the actual number of examples in each split.
2. Explain why the counts may not be exact integer percentages.
3. Verify that the three split sizes add back to the full dataset size.
4. Why must the test set remain untouched during model development?

**Your answers:**

In [ ]:
def split_class_report(name, labels):
    counts = labels.value_counts().sort_index()
    proportions = labels.value_counts(normalize=True).sort_index()

    report = pd.DataFrame({
        "class_name": [class_names[i] for i in counts.index],
        "count": counts.values,
        "proportion": proportions.values,
    })

    print(f"\n{name}")
    display(report)

split_class_report("Training split", y_train)
split_class_report("Validation split", y_valid)
split_class_report("Test split", y_test)

## Task 3.4 — Why Stratification?

Inspect the class proportions in the three split reports.

Answer:

1. Are the class proportions approximately similar?
2. What does `stratify=y` do in the first split?
3. What does `stratify=y_temp` do in the second split?
4. Why can stratification be useful for classification?
5. Would stratification by the target be meaningful in an ordinary regression problem? Explain.

**Your answers:**

## Task 3.5 — Check That the Splits Do Not Overlap

Each original row has an index.

Before running the next cell, predict the size of the intersections:

$$
Train \cap Validation,
\qquad
Train \cap Test,
\qquad
Validation \cap Test.
$$

**Your prediction:**

In [ ]:
train_ids = set(X_train.index)
valid_ids = set(X_valid.index)
test_ids = set(X_test.index)

print("Train ∩ Validation:", len(train_ids & valid_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(valid_ids & test_ids))

assert len(train_ids & valid_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(valid_ids & test_ids) == 0
assert len(train_ids | valid_ids | test_ids) == len(X)

print("Split integrity checks passed.")

# Part IV — Build a Majority-Class Baseline

A baseline gives us a minimum reference point.

For classification, a simple baseline is:

> Find the most frequent class **in the training set**, then predict that class for every new example.

The baseline does not use the input features $X$.

Its purpose is not to be intelligent. Its purpose is to answer:

> Does a future model actually improve over a trivial strategy?

## Task 4.1 — Predict the Baseline Class

Before implementing the baseline:

1. Which class do you expect to be the majority class in your **training split**?
2. What validation accuracy do you expect approximately?
3. Why should this baseline be fitted using `y_train` rather than `y_valid` or `y_test`?

**Your prediction:**

## Task 4.2 — Complete the Baseline

Complete the `fit` method.

The method must:

1. inspect the training labels;
2. find the most frequent class;
3. store it in `self.majority_class`;
4. return `self`.

Do **not** use the validation or test labels inside `fit`.

In [ ]:
class MajorityBaseline:
    def __init__(self):
        self.majority_class = None

    def fit(self, y_training):
        # TODO: Find the most frequent class in y_training.
        # Hint: np.unique(..., return_counts=True) may help.
        raise NotImplementedError("Complete MajorityBaseline.fit().")

    def predict(self, n_examples):
        if self.majority_class is None:
            raise RuntimeError("Call fit() before predict().")

        return np.full(n_examples, self.majority_class, dtype=int)

### Self-check

After completing `fit`, run the next cell.

If the assertion fails, inspect your logic rather than copying a replacement implementation.

In [ ]:
toy_baseline = MajorityBaseline().fit(np.array([0, 0, 1, 0, 2]))
assert toy_baseline.majority_class == 0
assert np.array_equal(toy_baseline.predict(4), np.array([0, 0, 0, 0]))

print("Baseline implementation tests passed.")

In [ ]:
baseline = MajorityBaseline().fit(y_train.to_numpy())

valid_pred = baseline.predict(len(y_valid))
test_pred = baseline.predict(len(y_test))

valid_accuracy = accuracy_score(y_valid, valid_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Training-set majority class ID:", baseline.majority_class)
print("Training-set majority class name:", class_names[baseline.majority_class])
print(f"Validation baseline accuracy: {valid_accuracy:.3f}")
print(f"Test baseline accuracy:       {test_accuracy:.3f}")

## Task 4.3 — Interpret the Baseline

Answer:

1. Which class did the baseline learn from the training labels?
2. What validation accuracy did it achieve?
3. What test accuracy did it achieve?
4. Why can validation and test baseline accuracies differ slightly?
5. Did the baseline use any sepal or petal measurements?
6. If a future classifier obtains only 1–2 percentage points above this baseline, would you automatically consider it useful? Why or why not?

**Your answers:**

## Task 4.4 — Training vs. Inference

The baseline contains a very small form of learning.

### Training

```text
training labels -> identify majority class -> store majority_class
```

### Inference

```text
stored majority_class + number of new examples -> predictions
```

Answer:

1. Which value is learned during `fit()`?
2. Does `predict()` change the learned value?
3. Why is training conceptually different from inference?
4. How will this distinction carry over to logistic regression, decision trees, and neural networks later?

**Your answers:**

# Part V — Detect Data Leakage and Invalid Evaluation

A model can appear excellent for the wrong reason.

**Data leakage** occurs when training or model development uses information that would not legitimately be available for future predictions.

You will diagnose three different leakage problems.

## Task 5.1 — Target Leakage

Suppose someone creates the following feature:

```text
confirmed_species_after_botanical_identification
```

It is created **after** the flower species has already been confirmed and therefore directly reveals whether the flower is *Iris virginica*.

The next cell demonstrates why this is invalid.

In [ ]:
X_leaky = X.copy()
X_leaky["confirmed_species_after_botanical_identification"] = y

# This is NOT a legitimate ML model.
# It simply reads the leaked answer.
leaked_predictions = X_leaky.loc[
    X_test.index,
    "confirmed_species_after_botanical_identification"
].to_numpy()

leaked_accuracy = accuracy_score(y_test, leaked_predictions)
print(f"Apparent test accuracy using leaked answer: {leaked_accuracy:.3f}")

## Task 5.2 — Explain the 100% Result

Answer:

1. Why is the apparent accuracy perfect?
2. Why is this not evidence of learning?
3. Would `confirmed_species_after_botanical_identification` be available **before** the model is asked to identify the flower?
4. What is the correct fix?
5. Give one target-leakage example from another application domain.

**Your answers:**

## Task 5.3 — Test-Set Tuning

A student tries five future models:

```text
Model A, Model B, Model C, Model D, Model E
```

They repeatedly evaluate all five on the test set and choose the one with the highest test accuracy.

Answer:

1. What is wrong with this procedure?
2. Which split should be used to choose among the five models?
3. After the choice is finalized, when should the test set be used?
4. Why does repeated test-set use make the final reported score less trustworthy?

**Your answers:**

## Task 5.4 — Split Contamination Debugging

The following code deliberately contaminates a validation set by copying five training rows into it.

Run the cell and inspect the overlap.

In [ ]:
X_valid_bad = pd.concat([X_valid, X_train.iloc[:5]], axis=0)
y_valid_bad = pd.concat([y_valid, y_train.iloc[:5]], axis=0)

overlap_bad = set(X_train.index) & set(X_valid_bad.index)

print("Number of training rows also present in BAD validation set:", len(overlap_bad))
print("Overlapping row IDs:", sorted(overlap_bad))

## Task 5.5 — Fix the Contamination

Create `X_valid_fixed` and `y_valid_fixed` so that they contain only the original uncontaminated validation data.

Then run the self-check.

Do not modify `X_train`, `y_train`, `X_test`, or `y_test`.

In [ ]:
# TODO: restore the clean validation split.
X_valid_fixed = None
y_valid_fixed = None

In [ ]:
assert X_valid_fixed is not None and y_valid_fixed is not None
assert len(set(X_train.index) & set(X_valid_fixed.index)) == 0
assert len(X_valid_fixed) == len(X_valid)
assert len(y_valid_fixed) == len(y_valid)

print("Validation contamination fixed.")

# Part VI — Experiment: What Changes When the Random Seed Changes?

A train/validation/test split is a sample from the available dataset.

Changing the random seed changes **which observations** enter each split while preserving the intended proportions.

You will compare your personalized split with a second split generated from:

$$
SEED+1.
$$

## Task 6.1 — Predict Before Running

Before running the experiment, predict:

1. Will the split sizes change?
2. Will the exact row membership change?
3. Will the class proportions remain approximately similar?
4. Will the majority baseline accuracy necessarily be exactly identical?
5. Should we choose a random seed simply because it gives the highest validation score?

**Your prediction and reasoning:**

In [ ]:
ALT_SEED = SEED + 1

X_train_2, X_temp_2, y_train_2, y_temp_2 = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=ALT_SEED,
    stratify=y,
)

X_valid_2, X_test_2, y_valid_2, y_test_2 = train_test_split(
    X_temp_2,
    y_temp_2,
    test_size=0.50,
    random_state=ALT_SEED,
    stratify=y_temp_2,
)

baseline_2 = MajorityBaseline().fit(y_train_2.to_numpy())
valid_pred_2 = baseline_2.predict(len(y_valid_2))

print("Original seed:", SEED)
print("Alternative seed:", ALT_SEED)
print()
print("Original validation baseline accuracy:", round(valid_accuracy, 3))
print("Alternative validation baseline accuracy:", round(accuracy_score(y_valid_2, valid_pred_2), 3))
print()
print("Number of observations shared by the two training sets:",
      len(set(X_train.index) & set(X_train_2.index)))
print("Original training size:", len(X_train))
print("Alternative training size:", len(X_train_2))

## Task 6.2 — Analyze the Experiment

Compare the result with your prediction.

Answer:

1. Which quantities stayed the same?
2. Which quantities changed?
3. Why does changing the random seed change row membership?
4. Why do we use fixed seeds in experiments?
5. Why would searching many seeds for the best validation result be a questionable practice?
6. What more robust approach will later help reduce dependence on a single split?

**Your answers:**

> Hint for Question 6: later we will use **cross-validation**.

# Part VII — Personalized Formulation Challenge

Your student-ID seed assigns one of four scenarios.

Run the next cell to reveal your scenario.

You must complete the formulation **individually**.

In [ ]:
scenarios = [
    {
        "title": "Campus Shuttle Delay",
        "goal": "Predict the shuttle delay in minutes before a bus arrives.",
        "available_data": [
            "route", "time of day", "day of week",
            "current traffic level", "weather", "actual arrival time"
        ],
    },
    {
        "title": "Library User Segmentation",
        "goal": "Discover groups of library users based on usage behavior.",
        "available_data": [
            "visits per month", "books borrowed", "digital-resource sessions",
            "average visit duration", "department"
        ],
    },
    {
        "title": "Course At-Risk Prediction",
        "goal": "Predict in Week 5 whether a student is at risk of failing the course.",
        "available_data": [
            "attendance through Week 5", "quiz scores through Week 5",
            "LMS activity through Week 5", "final exam score", "final course result"
        ],
    },
    {
        "title": "Apartment Rental Price",
        "goal": "Predict monthly rental price before a new apartment listing is published.",
        "available_data": [
            "area", "district", "number of rooms",
            "building age", "furnished status", "published monthly rent"
        ],
    },
]

scenario_index = SEED % len(scenarios)
assigned = scenarios[scenario_index]

print("Your assigned scenario:", assigned["title"])
print("Goal:", assigned["goal"])
print("Available data:")
for item in assigned["available_data"]:
    print(" -", item)

## Task 7.1 — Complete the ML Problem Specification

For your assigned scenario, complete:

### A. Learning formulation

- **Learning paradigm:**  
- **Classification / regression / clustering / other:**  
- **One observation represents:**  
- **Target $y$ (if any):**  
- **Input features $X$:**  
- **Potential feature that must be excluded:**  
- **Reason for exclusion:**  

### B. Evaluation design

- **Training-set role:**  
- **Validation-set role:**  
- **Test-set role:**  
- **Simple baseline:**  
- **One appropriate evaluation measure or evaluation principle:**  

### C. Generalization question

In 2–4 sentences, explain what it would mean for a model in your scenario to **generalize** successfully.

## Task 7.2 — Prediction Before Modification

Choose **one** of the following modifications to your assigned scenario:

- remove an informative feature;
- add a leaked feature;
- make one class much rarer;
- reduce the dataset size substantially;
- change the prediction time so that fewer features are available.

Before changing your formulation, predict:

1. which part of the ML workflow is affected;
2. whether expected model performance becomes better, worse, or misleading;
3. why.

**Your prediction:**

## Task 7.3 — Modify and Defend

Write the modified problem specification in 4–8 lines.

Then explain:

- what changed;
- whether the original evaluation strategy is still valid;
- whether the baseline should change;
- one risk introduced by the modification.

**Your answer:**

# Part VIII — Individual Understanding Check

Your instructor may select **one** of the following for a 60–90 second individual explanation.

1. Why do we need three different roles for training, validation, and test data?
2. Explain why a majority-class baseline can be useful even though it ignores all features.
3. Give an example of target leakage and explain exactly why it invalidates evaluation.
4. Explain what stratification changes and what it does **not** change.
5. Why should the final test set be used only after model selection?
6. In your personalized scenario, identify $X$, $y$, and the prediction time.

> You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What is the most important decision that must be made **before** choosing an ML algorithm?
2. What did the majority baseline teach you about evaluation?
3. Which leakage example in this lab was most serious, and why?
4. What is one reason two students can obtain slightly different validation results even when both workflows are correct?
5. What concept from this lab do you expect to reuse in every later ML lab?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] completed scenario-classification table;
- [ ] feature/target reasoning;
- [ ] your own student-ID-derived split seed;
- [ ] predictions written **before** execution where requested;
- [ ] train/validation/test split analysis;
- [ ] completed `MajorityBaseline.fit()` implementation;
- [ ] baseline validation and test results;
- [ ] leakage explanations;
- [ ] corrected contaminated validation split;
- [ ] alternative-seed experiment and interpretation;
- [ ] personalized problem formulation;
- [ ] modification challenge;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

Save the final notebook with a clear filename before submission.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Prediction / debugging evidence | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require more than correct output.

A submission should demonstrate that you understand:

$$
\text{problem formulation}
\rightarrow
\text{data roles}
\rightarrow
\text{baseline}
\rightarrow
\text{valid evaluation}.
$$

# Lab 1 Summary

You should now be able to explain the complete foundation of a machine-learning experiment:

$$
\boxed{
\text{Define the task}
\rightarrow
\text{Identify }X\text{ and }y
\rightarrow
\text{Split correctly}
\rightarrow
\text{Establish a baseline}
\rightarrow
\text{Validate}
\rightarrow
\text{Test once}
}
$$

### Key lessons

- A model cannot compensate for a badly formulated problem.
- Training, validation, and test data have different roles.
- A baseline tells us whether later models add value.
- High performance can be meaningless when caused by leakage.
- Reproducible experiments require controlled splits and clear reasoning.

**Next lab:** Exploratory Data Analysis and Data Quality.